In [1]:
import glob
import json
import os
import sys
import time
from collections import deque
from pathlib import Path

import cudf
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import seaborn as sns
import shap
import wandb
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

sys.path.append(os.path.abspath(".."))

from src.utils.print_duration import print_duration
import src.models.xgb.xgb_cv_trainer2 as cv

In [2]:
import importlib
importlib.reload(cv)

<module 'src.models.xgb.xgb_cv_trainer2' from '/home/hanse/kaggle/binary-bank/src/models/xgb/xgb_cv_trainer2.py'>

In [3]:
# Configuration
DATA_ID = "026"
base_dir = f"../artifacts/features/base/{DATA_ID}"

In [4]:
trainer = cv.XGBCVTrainer(
    DATA_ID,
    base_dir
)
score = trainer.fit_one_fold()

CATS: 0 columns
[0]	train-auc:0.96258	eval-auc:0.96139
[100]	train-auc:0.97945	eval-auc:0.97362
[200]	train-auc:0.98364	eval-auc:0.97403
[300]	train-auc:0.98659	eval-auc:0.97407
[400]	train-auc:0.98925	eval-auc:0.97402
[458]	train-auc:0.99065	eval-auc:0.97401

QuantileDMatrix Build Time: 00:00:30
Training Time: 00:00:44

Train AUC: 0.98542
Valid AUC: 0.97408
Total Runtime: 00:01:15


In [ ]:
class CFG:
    COMPETITION = "binary-bank"
    DEBUG = False
    TUNING = False
    MODEL = "xgb"
    DATA_ID = "026"
    SEED = 42

In [6]:
train_path = base_dir + "/tr_df026-seed42.parquet"

train_rows = pq.ParquetFile(train_path).metadata.num_rows
print(train_rows)
oof_preds = np.zeros((train_rows))
print(oof_preds.shape)

795211
(795211,)


In [7]:
pf = pq.ParquetFile(train_path)

# スキーマからカラム名を取得
columns = pf.schema.names

# row_id が含まれているか確認
print("row_id" in columns)

True


In [ ]:
def class2dict(f):
    return dict(
        (name, getattr(f, name)) for name in dir(f) if not name.startswith("__")
    )

In [11]:
import psutil
import resource

In [16]:
rss = psutil.Process(os.getpid()).memory_info().rss
print(rss)
def pretty(b):
    for u in ["B","KB","MB","GB","TB"]:
        if b < 1024 or u == "TB":
            return f"{b:.2f} {u}"
        b /= 1024

_pretty_bytes(rss)

1027170304


'979.59 MB'

In [14]:
vm = psutil.virtual_memory()
print(f"WSL memory  used={vm.used/1024**3:.2f} GB  "
      f"avail={vm.available/1024**3:.2f} GB  total={vm.total/1024**3:.2f} GB")

WSL memory  used=1.84 GB  avail=21.28 GB  total=23.47 GB


In [17]:
ru = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
peak_bytes = ru if sys.platform == "darwin" else ru * 1024  # macはbytes, LinuxはKB
print("Process peak RSS:", pretty(peak_bytes))

Process peak RSS: 10.09 GB


In [ ]:
def read_proc_status():
    rss = hwm = None
    try:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    rss = int(line.split()[1]) * 1024      # kB → bytes
                elif line.startswith("VmHWM:"):
                    hwm = int(line.split()[1]) * 1024      # kB → bytes
        return rss, hwm
    except FileNotFoundError:
        # 稀に /proc がマウントされていない環境用のフォールバック
        return None, None

In [ ]:
def _pretty_bytes(b):
    for u in ["B", "KB", "MB", "GB", "TB"]:
        if b < 1024 or u == "TB":
            return f"{b:.2f} {u}"
        b /= 1024

def cpu_mem_now_and_peak():
    """returns (rss_bytes_now, peak_rss_bytes_since_process_start)"""
    # now
    if psutil:
        rss = psutil.Process(os.getpid()).memory_info().rss
    else:
        # フォールバック（Linux）
        try:
            with open("/proc/self/statm") as f:
                pages = int(f.read().split()[1])
            rss = pages * os.sysconf("SC_PAGE_SIZE")
        except Exception:
            rss = -1

    # peak
    peak = -1
    if resource:
        ru = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        # LinuxはKB, macOSはbytes
        peak = ru if sys.platform == "darwin" else ru * 1024
    else:
        # Linuxのみ: /proc/self/status の VmHWM を読む
        try:
            for line in open("/proc/self/status"):
                if line.startswith("VmHWM:"):
                    peak = int(line.split()[1]) * 1024
                    break
        except Exception:
            pass
    return rss, peak

def gpu_mem_now(device_index=0):
    """returns (used_bytes, total_bytes) via NVML if available; else tries CuPy; else None"""
    # NVML (推奨)
    try:
        import pynvml
        pynvml.nvmlInit()
        h = pynvml.nvmlDeviceGetHandleByIndex(device_index)
        m = pynvml.nvmlDeviceGetMemoryInfo(h)
        return int(m.used), int(m.total)
    except Exception:
        pass
    # CuPy (全体ではなく“自プロセスのメモリプール”の占有状況)
    try:
        import cupy as cp
        pool = cp.get_default_memory_pool()
        used = pool.used_bytes()     # プールで現在使用中
        total = pool.total_bytes()   # プールが確保済み（解放待ち含む）
        return int(used), int(total) if total > 0 else None
    except Exception:
        return None

def print_mem(tag=""):
    rss, peak = cpu_mem_now_and_peak()
    gpu = gpu_mem_now()
    out = [f"[MEM] {tag}  CPU now: {_pretty_bytes(rss)}"]
    if peak >= 0:
        out.append(f"CPU peak: {_pretty_bytes(peak)}")
    if gpu is not None:
        used, total = gpu
        # NVMLなら total はGPU総メモリ。CuPy fallback時は「確保済み」を total に入れていることに注意。
        if total:
            out.append(f"GPU: {_pretty_bytes(used)} / {_pretty_bytes(total)}")
        else:
            out.append(f"GPU used: {_pretty_bytes(used)}")
    print(" | ".join(out))